In [ ]:
# 📦 安装核心依赖包
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 #colab一般都有
!pip install numpy matplotlib pandas
!pip install GPUtil  # GPU监控
# 🚀 可选：增强功能包
!pip install tensorboard  # 训练可视化
!pip install tqdm         # 进度条美化
!pip install seaborn      # 高级绘图

In [ ]:
import GPUtil

def get_nvidia_gpu_info():
    """
    获取NVIDIA显卡信息（型号、显存、使用率）
    :return: 列表，每个元素为显卡的详细信息
    """
    gpus = GPUtil.getGPUs()
    if not gpus:
        return None
    gpu_info = []
    for gpu in gpus:
        gpu_info.append({
            "显卡ID": gpu.id,
            "型号": gpu.name,
            "总显存(GB)": round(gpu.memoryTotal, 2),
            "已用显存(GB)": round(gpu.memoryUsed, 2),
            "空闲显存(GB)": round(gpu.memoryFree, 2),
            "显卡使用率(%)": gpu.load * 100,
            "温度(℃)": gpu.temperature
        })
    return gpu_info

# 测试
if __name__ == "__main__":
    print("=== NVIDIA显卡信息 ===")
    nvidia_gpu = get_nvidia_gpu_info()
    if nvidia_gpu:
        for idx, gpu in enumerate(nvidia_gpu, 1):
            print(f"\n显卡{idx}:")
            for key, value in gpu.items():
                print(f"  {key}: {value}")
    else:
        print("未检测到NVIDIA显卡")


In [ ]:
#td3.py
import copy
import numpy as np
import torch
import torch.nn.functional as F
import torch.optim as optim

# ==========================================
# 核心修复 1：在文件开头定义全局设备 (GPU/CPU)
# 这样下面的所有数据都可以知道往哪里搬
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class TD3:
    def __init__(self, state_dim, action_dim, max_action):
        self.max_action = max_action

        # 初始化 Actor 和 Critic 网络
        self.actor = Actor(state_dim, action_dim, max_action).to(device)
        self.actor_target = copy.deepcopy(self.actor).to(device)

        self.critic = Critic(state_dim, action_dim).to(device)
        self.critic_target = copy.deepcopy(self.critic).to(device)

        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=3e-4)
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=3e-4)

        self.discount = 0.99
        self.tau = 0.005
        self.policy_noise = 0.2
        self.noise_clip = 0.5
        self.policy_freq = 2
        self.total_it = 0

        self.actor_old = copy.deepcopy(self.actor).to(device)
        self.anchor_coeff = 0.0

    def select_action(self, state, exploration_noise=0.0):
        # ==========================================
        # 核心修复 2：将输入的 numpy state 变成 Tensor 后，
        # 必须紧跟 .to(device) 把它推到 GPU 上！
        # ==========================================
        state = torch.FloatTensor(state.reshape(1, -1)).to(device)

        # 模型在 GPU 里算完动作后，【先 .cpu() 搬回来】，再去 .numpy()
        action = self.actor(state).cpu().data.numpy().flatten()

        if exploration_noise != 0.0:
            noise = np.random.normal(0, exploration_noise, size=action.shape)
            action = action + noise

        return np.clip(action, -self.max_action, self.max_action)

    def update(self, replay_buffer, batch_size=100):
        self.total_it += 1

        # 从经验回放中采样 (这里拿到的数据已经在 GPU 上了，因为我们在 Buffer 里修改了)
        states, actions, rewards, next_states, dones = replay_buffer.sample(batch_size)

        # Critic网络更新
        with torch.no_grad():
            noise = (torch.randn_like(actions) * self.policy_noise).clamp(-self.noise_clip, self.noise_clip)
            next_actions = (self.actor_target(next_states) + noise).clamp(-self.max_action, self.max_action)

            target_q1, target_q2 = self.critic_target(next_states, next_actions)
            target_q = torch.min(target_q1, target_q2)
            target_q = rewards + (1 - dones) * self.discount * target_q

        current_q1, current_q2 = self.critic(states, actions)
        critic_loss = F.mse_loss(current_q1, target_q) + F.mse_loss(current_q2, target_q)

        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        self.critic_optimizer.step()

        # Actor 更新逻辑
        if self.total_it % self.policy_freq == 0:
            current_actions = self.actor(states)
            q1, _ = self.critic(states, current_actions)
            actor_loss = -q1.mean()

            # 策略锚定损失 (Behavioral Constraining)
            if self.anchor_coeff > 0:
                with torch.no_grad():
                    old_actions = self.actor_old(states)
                anchor_loss = F.mse_loss(current_actions, old_actions)
                actor_loss += self.anchor_coeff * anchor_loss

            self.actor_optimizer.zero_grad()
            actor_loss.backward()
            self.actor_optimizer.step()

            # 更新目标网络
            for param, target_param in zip(self.critic.parameters(), self.critic_target.parameters()):
                target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)

            for param, target_param in zip(self.actor.parameters(), self.actor_target.parameters()):
                target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)

    def sync_old_policy(self):
        self.actor_old.load_state_dict(self.actor.state_dict())


class ReplayBuffer:
    def __init__(self, capacity=100000):
        self.capacity = capacity
        self.buffer = []
        self.position = 0

    def add(self, state, action, reward, next_state, done):
        if len(self.buffer) < self.capacity:
            self.buffer.append(None)
        self.buffer[self.position] = (state, action, reward, next_state, done)
        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        indices = np.random.choice(len(self.buffer), batch_size)
        samples = [self.buffer[i] for i in indices]

        states = np.array([s[0] for s in samples])
        actions = np.array([s[1] for s in samples])
        rewards = np.array([s[2] for s in samples])
        next_states = np.array([s[3] for s in samples])
        dones = np.array([s[4] for s in samples])

        # ==========================================
        # 核心修复 3：从 Buffer 里抽样出来准备给模型训练的数据，
        # 必须全部加上 .to(device) 扔到 GPU 里去！
        # ==========================================
        return (
            torch.FloatTensor(states).to(device),
            torch.FloatTensor(actions).to(device),
            torch.FloatTensor(rewards).unsqueeze(1).to(device),
            torch.FloatTensor(next_states).to(device),
            torch.FloatTensor(dones).unsqueeze(1).to(device)
        )

    def __len__(self):
        return len(self.buffer)

In [ ]:
#model.py
import torch
import torch.nn as nn

class Actor(nn.Module):
    def __init__(self, state_dim, action_dim, max_action):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 256),  # 输入层改为6维？
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, action_dim),
            nn.Tanh()
        )
        self.max_action = max_action

    def forward(self, state):
        return self.net(state) * self.max_action

class Critic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.q1 = nn.Sequential(
            nn.Linear(state_dim + action_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

        self.q2 = nn.Sequential(
            nn.Linear(state_dim + action_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward(self, state, action):
        sa = torch.cat([state, action], 1)
        return self.q1(sa), self.q2(sa)

In [ ]:
#environment.py
import numpy as np

class DroneEnv:
    """
    无人机路径规划仿真环境
    状态空间：6 or 14维 [x,y,z, dx,dy,dz] (当前位置 + 目标方向向量)
    动作空间：3维 [vx,vy,vz] (三维速度向量)
    """

    def __init__(self, max_episodes=5000):
        self.max_episodes = max_episodes # 将总回合数存入类属性
        # 环境维度参数
        self.action_dim = 3  # 动作空间维度（三维速度）
        # self.state_dim = 6   # 原来的状态空间维度（3D位置 + 3D目标方向）

        self.state_dim = 32 #状态维度修改，26条射线，加上6维位置方向一共32【2026年3月27日】
        # self.state_dim = 14
        self.step_count = 0  # 当前步数计数器
        self.max_step = 200  # 单回合最大步数
        self.prev_distance = 0.0 # 用于记录上一步的距离
        # ================= 新增：雷达射线方向初始化 =================
        # 增加射线数量，至少覆盖 26 个方向（这里AI修改的，我已经开始看不懂了）【2026年3月27日】
        dirs = []
        for x in [-1, 0, 1]:
            for y in [-1, 0, 1]:
                for z in [-1, 0, 1]:
                    if x == 0 and y == 0 and z == 0:
                        continue
                    dirs.append([x, y, z])
        dirs = np.array(dirs, dtype=np.float32)
        self.ray_dirs = dirs / np.linalg.norm(dirs, axis=1, keepdims=True)

        # dirs = np.array([
        #     [1, 1, 1], [1, 1, -1], [1, -1, 1], [1, -1, -1],
        #     [-1, 1, 1], [-1, 1, -1], [-1, -1, 1], [-1, -1, -1]
        # ], dtype=np.float32)
        # 将方向向量归一化（长度变成1）
        self.ray_dirs = dirs / np.linalg.norm(dirs, axis=1, keepdims=True)

        self.max_ray_length = 5.0  # 雷达最大探测距离为 5 米
        # ==========================================================
    def _get_lidar_data(self):
        """
        计算8条射线到最近障碍物的距离。
        返回：长度为8的数组，数值经过归一化 (0~1)。1表示安全(没扫到东西)，0表示贴脸。
        """
        distances = np.full(len(self.ray_dirs), self.max_ray_length)

        for i, ray in enumerate(self.ray_dirs):
            min_t = self.max_ray_length

            for obs in self.obstacles:
                # 射线与球体求交点的数学计算 (一元二次方程)
                # 射线公式: P = origin + t * ray
                # 球体公式: ||P - center||^2 = radius^2
                oc = self.position - obs['pos']
                b = 2.0 * np.dot(ray, oc)
                c = np.dot(oc, oc) - obs['radius']**2

                discriminant = b**2 - 4*c  # 判别式 (b^2 - 4ac, a=1因为ray已归一化)

                if discriminant > 0:
                    # 有交点，计算距离 t
                    t1 = (-b - np.sqrt(discriminant)) / 2.0
                    t2 = (-b + np.sqrt(discriminant)) / 2.0

                    # 取大于0且最小的 t (即射线前方最近的交点)
                    if 0 < t1 < min_t:
                        min_t = t1
                    elif 0 < t2 < min_t:
                        min_t = t2

            distances[i] = min_t

        # 归一化测距数据：距离 / 最大探测距离
        return distances / self.max_ray_length
    def reset(self, current_difficulty=1.0):
        self.step_count = 0
        self.position = np.random.uniform(-5, 5, size=3)
        self.target = self.position + np.random.uniform(-8, 8, size=3)
        self.target = np.clip(self.target, -14, 14)
        self.target[2] = np.abs(self.target[2]) + 1.0

        self.prev_distance = np.linalg.norm(self.target - self.position)

        # ================= 新增：课程学习逻辑 =================
        self.obstacles = []
        #【2026年3月27日修改引入课程学习】

         # 1. 动态增加数量：从 0 增加到 12 个
        num_obstacles = int(current_difficulty * 12)

        # 2. 障碍物进化：难度越高，障碍物越小但越密集（制造窄缝）
        for _ in range(num_obstacles):
            # 难度越高，障碍物越倾向于出现在起始点和目标点的中间连线上
            alpha = np.random.uniform(0.2, 0.8)
            base_pos = self.position + alpha * (self.target - self.position)

            # 增加扰动，制造错位感
            obs_pos = base_pos + np.random.uniform(-4.0, 4.0, size=3)

            # 核心：高难度下半径减小但数量增多，形成类似“森林”的遮挡
            radius = np.random.uniform(0.5, 1.5 - (0.5 * current_difficulty))

            # 3. 动态属性：高难度下赋予速度
            vel = np.random.uniform(-0.1, 0.1, size=3) * (current_difficulty > 0.8)

            self.obstacles.append({'pos': obs_pos, 'radius': radius, 'vel': vel})

        return self._get_state()


    def _get_state(self):
        """
        生成当前状态向量并进行归一化
        """
        direction = self.target - self.position
        # 将位置和方向都归一化到约 [-1, 1] 范围内 (最大边界为15)
        norm_position = self.position / 15.0
        norm_direction = direction / 15.0
        # 获取雷达数据
        lidar_data = self._get_lidar_data()

        # 【状态拼接】：现在的状态包含了 [位置(3), 目标方向(3), 雷达测距(8)]
        return np.concatenate([norm_position, norm_direction, lidar_data])
    def step(self, action):
        # 1. 物理模拟
        self.position += action * 0.3
        self.step_count += 1
        target_distance = np.linalg.norm(self.target - self.position)

        done = False
        reward = 0.0

        # 【新增】：初始化 info 字典
        info = {'TimeLimit.truncated': False, 'is_success': False,
             'is_success': False, 'is_collision': False}

        # --- 碰撞检测 ---
        collision = False
        min_obs_dist = float('inf')
        for obs in self.obstacles:
            dist_to_obs = np.linalg.norm(self.position - obs['pos'])
            min_obs_dist = min(min_obs_dist, dist_to_obs)
            if dist_to_obs < (obs['radius'] + 0.2):
                collision = True
                break

        if collision:
            reward = -50.0  # 【建议调小惩罚，原来是-200】
            done = True
            info['is_collision'] = True
            return self._get_state(), reward, done, info  # 返回 info

        # --- 到达目标检测 ---
        if target_distance < 1.5:
            reward = 100.0
            done = True
            info['is_success'] = True  # 记录成功

        # --- 边界碰撞检测 ---
        elif np.any(np.abs(self.position) > 15):
            reward = -50.0
            done = True

        # --- 正常飞行 ---
        else:
            progress_reward = (self.prev_distance - target_distance) * 10.0
            action_penalty = 0.05 * np.linalg.norm(action)

            repulsion_penalty = 0.0
            for obs in self.obstacles:
                dist_to_obs = np.linalg.norm(self.position - obs['pos'])
                safe_margin = obs['radius'] + 1.0
                if dist_to_obs < safe_margin:
                    repulsion_penalty += 2.0 * (safe_margin - dist_to_obs)

            reward = progress_reward - action_penalty - repulsion_penalty

        # 【2026年3月27日修改】：超时检测
        if not done and self.step_count >= self.max_step:
            done = True
            info['TimeLimit.truncated'] = True  # 标记这是一个超时引起的结束

        self.prev_distance = target_distance
        return self._get_state(), reward, done, info  # 确保返回四个参数


In [ ]:
#train.py
import os
import csv
import torch
import random
import numpy as np
from collections import deque
from pathlib import Path

# ==========================================
# 1. 严格的随机种子设置函数（RL复现核心）
# ==========================================
def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

# ==========================================
# 2. 封装核心训练逻辑为独立函数
# ==========================================
def run_experiment(use_curriculum, seed, max_episodes=5000):
    print(f"\n{'='*50}")
    print(f"🚀 开始实验 | 课程学习: {use_curriculum} | 随机种子: {seed}")
    print(f"{'='*50}")

    # --- 1. 初始化设置 ---
    set_seed(seed)

    # 动态构建干净的保存路径
    mode_name = "Curriculum" if use_curriculum else "No_Curriculum"
    output_dir = f"./experiments/{mode_name}/seed_{seed}"
    os.makedirs(output_dir, exist_ok=True)
    log_file = os.path.join(output_dir, 'training_log.csv')

    # 初始化 CSV 日志表头
    with open(log_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['episode', 'reward', 'steps', 'success', 'collision', 'difficulty'])

    # 初始化环境与 Agent (确保每次传入新环境)
    env = DroneEnv(max_episodes=max_episodes)
    state_dim = env.state_dim
    action_dim = env.action_dim
    max_action = 1.0

    agent = TD3(state_dim, action_dim, max_action)

    # 如果检测到 GPU，确保模型搬到 GPU（修复你之前的隐患）
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    agent.actor.to(device)
    agent.actor_target.to(device)
    agent.critic.to(device)
    agent.critic_target.to(device)
    agent.actor_old.to(device)

    replay_buffer = ReplayBuffer(capacity=100000)

    # 训练参数设定
    current_noise = 0.2
    min_noise = 0.01
    decay_rate = 0.999
    success_history = deque(maxlen=100)

    # 难度初始化
    current_difficulty = 0.1 if use_curriculum else 1.0
    upgrade_threshold = 0.85  # 成功率达到 85% 升班

    # --- 2. 开始训练循环 ---
    for episode in range(max_episodes):
        state = env.reset(current_difficulty=current_difficulty)
        episode_reward = 0

        # 课程学习独有的策略锚定逻辑
        if use_curriculum:
            if episode in [1000, 2000, 3000]:
                print(f"   [Episode {episode}] 进入新课程阶段，开启策略锚定")
                agent.sync_old_policy()
                agent.anchor_coeff = 1.0
            agent.anchor_coeff *= 0.995
        else:
            agent.anchor_coeff = 0.0

        for t in range(env.max_step):
            action = agent.select_action(state, exploration_noise=current_noise)
            next_state, reward, done, info = env.step(action)

            # 超时处理
            done_bool = float(done) if not info.get('TimeLimit.truncated', False) else 0.0
            replay_buffer.add(state, action, reward, next_state, done_bool)

            if len(replay_buffer) > 1000:
                agent.update(replay_buffer)

            state = next_state
            episode_reward += reward

            if done:
                is_success = 1 if info.get('is_success', False) else 0
                is_collision = 1 if info.get('is_collision', False) else 0

                # 记录日志
                with open(log_file, 'a', newline='') as f:
                    writer = csv.writer(f)
                    writer.writerow([episode, round(episode_reward, 2), env.step_count, is_success, is_collision, current_difficulty])
                break

        # 更新噪声
        current_noise = max(min_noise, current_noise * decay_rate)

        # 打印进度 (每100回合打印一次，防止Colab输出卡死)
        if episode % 100 == 0:
            print(f"   Episode {episode} | Reward: {episode_reward:.2f} | Noise: {current_noise:.3f} | Diff: {current_difficulty:.2f}")

        # 难度升级逻辑 (仅课程学习)
        success_history.append(1 if info.get('is_success', False) else 0)
        avg_success = sum(success_history) / len(success_history) if success_history else 0

        if use_curriculum and len(success_history) == 100 and avg_success > upgrade_threshold:
            if current_difficulty < 1.0:
                current_difficulty = min(1.0, current_difficulty + 0.1)
                print(f"   ⭐ 难度升级！当前难度: {current_difficulty:.2f} | 策略锚定同步")
                agent.sync_old_policy()
                success_history.clear()

    # --- 3. 训练结束，保存最终模型 ---
    torch.save(agent.actor.state_dict(), os.path.join(output_dir, "td3_actor_final.pth"))
    torch.save(agent.critic.state_dict(), os.path.join(output_dir, "td3_critic_final.pth"))
    print(f"✅ 实验完成! 模型和日志已保存至: {output_dir}")

In [ ]:
# 定义你要跑的随机种子（RL界常用的吉祥数）
seeds = [42, 1024, 2026]

# 遍历两种条件：课程学习 (True) 和 无课程学习 (False)
for use_cl in [True, False]:
    for seed in seeds:
        # 这里 max_episodes 可以先改成 100 测试一下流程，没问题再改回 5000
        run_experiment(use_curriculum=use_cl, seed=seed, max_episodes=5000)

print("\n🎉 所有消融实验已全部运行完毕！请去左侧文件夹查看 ./experiments 目录。")

In [ ]:
#analysis.py科研绘图版，“嘻嘻，不能让实验数据影响了我的结论”
import pandas as pd
import matplotlib.pyplot as plt
import os
import argparse
import sys
import glob
import seaborn as sns

# ==========================================
# 1. 修正路径参数：指向真实的训练输出目录
# ==========================================
parser = argparse.ArgumentParser()
parser.add_argument('--runs_dir', default='./experiments', help='包含所有实验日志的根目录')
parser.add_argument('--output_dir', default='./analysis_results_Plz_Converge', help='分析图表保存目录')

if 'ipykernel' in sys.modules:
    args = parser.parse_args(args=[])
else:
    args = parser.parse_args()

os.makedirs(args.output_dir, exist_ok=True)

# 导师建议的合理平滑参数（不要太大，保留真实方差）
INITIAL_RUN_SMOOTH_WINDOW = 10
FINAL_AVERAGE_SMOOTH_WINDOW = 50
RATE_SMOOTH_WINDOW = 50

# ==========================================
# 2. 递归读取多级目录下的 CSV 并打标签 (核心修改)
# ==========================================
all_data = []
# 使用 recursive=True 遍历例如 ./experiments/Curriculum/seed_42/training_log.csv
csv_files = glob.glob(os.path.join(args.runs_dir, '**', 'training_log.csv'), recursive=True)

for file_path in csv_files:
    df = pd.read_csv(file_path)

    # 从路径中聪明地解析出实验条件和种子
    # 路径示例: ./experiments/Curriculum/seed_42/training_log.csv
    path_parts = os.path.normpath(file_path).split(os.sep)
    condition = path_parts[-3] # 'Curriculum' 或 'No_Curriculum'
    seed = path_parts[-2]      # 'seed_42'

    df['Condition'] = condition
    df['Seed'] = seed
    df['run_id'] = f"{condition}_{seed}" # 唯一标识符

    # 计算平滑数据
    df['success_rate_run'] = df['success'].rolling(window=RATE_SMOOTH_WINDOW, min_periods=1).mean()
    df['collision_rate_run'] = df['collision'].rolling(window=RATE_SMOOTH_WINDOW, min_periods=1).mean()
    df['smoothed_reward_run'] = df['reward'].rolling(window=INITIAL_RUN_SMOOTH_WINDOW, min_periods=1).mean()

    all_data.append(df)

if not all_data:
    print(f"❌ 错误: 在 {args.runs_dir} 目录下没有找到任何日志！请先运行训练代码。")
    sys.exit()

combined_df = pd.concat(all_data, ignore_index=True)
print(f"✅ 成功加载了 {len(all_data)} 个实验运行的数据！")

# =====================================================================
# 3. 核心修复：按 Episode 和 Condition 双重分组计算统计量
# =====================================================================

# 同时计算均值和标准差
stats_mean = combined_df.groupby(['episode', 'Condition'])[['smoothed_reward_run', 'success_rate_run', 'collision_rate_run']].mean().reset_index()
stats_std = combined_df.groupby(['episode', 'Condition'])[['smoothed_reward_run', 'success_rate_run', 'collision_rate_run']].std().reset_index()

# 提取课程学习组的难度升级点（仅用于绘图参考）
upgrade_episodes = []
if 'difficulty' in combined_df.columns:
    # 仅从 Curriculum 组提取升级节奏
    cl_data = combined_df[combined_df['Condition'] == 'Curriculum']
    if not cl_data.empty:
        mean_diff = cl_data.groupby('episode')['difficulty'].mean().reset_index()
        mean_diff['diff_change'] = mean_diff['difficulty'].diff()
        upgrade_episodes = mean_diff[mean_diff['diff_change'] > 0.02]['episode'].tolist()
        print(f"检测到课程学习组的难度升级点: {upgrade_episodes}")

# =====================================================================
# 4. 绘图逻辑更新：支持对比对比
# =====================================================================

# --- A. 绘制平均奖励对比图 (average_reward_curve.png) ---
plt.figure(figsize=(12, 8))
sns.lineplot(data=combined_df, x='episode', y='smoothed_reward_run', hue='Condition', ci='sd')

# 绘制难度升级竖线
for i, up_ep in enumerate(upgrade_episodes):
    label = 'Difficulty Upgrade' if i == 0 else ""
    plt.axvline(x=up_ep, color='gray', linestyle='--', alpha=0.5, label=label)
    plt.text(up_ep + 20, plt.ylim()[1] * 0.9, f'Level Up', color='dimgray', rotation=90)

plt.title('Average Reward: Curriculum vs No_Curriculum (with SD Band)')
plt.xlabel('Episode')
plt.ylabel('Smoothed Reward')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(args.output_dir, 'average_reward_curve.png'))
plt.close()

# --- B. 封装对比率曲线绘图函数 ---
def plot_comparison_rate(df, y_col, title, filename, upgrade_eps):
    plt.figure(figsize=(12, 7))
    # 使用 seaborn 自动处理 hue 对比和阴影
    sns.lineplot(data=df, x='episode', y=y_col, hue='Condition', ci='sd')

    for up_ep in upgrade_eps:
        plt.axvline(x=up_ep, color='gray', linestyle='--', alpha=0.4)
        plt.text(up_ep + 10, 0.85, 'Level Up', color='gray', rotation=90, fontsize=9)

    plt.ylim(-0.05, 1.05)
    plt.title(title)
    plt.xlabel('Episode')
    plt.ylabel('Rate (0.0 - 1.0)')
    plt.grid(True, alpha=0.2)
    plt.legend(loc='upper left')
    plt.tight_layout()
    plt.savefig(os.path.join(args.output_dir, filename))
    plt.close()

# 执行成功率和碰撞率对比图绘制
plot_comparison_rate(combined_df, 'success_rate_run', 'Success Rate Comparison', 'success_rate_curve.png', upgrade_episodes)
plot_comparison_rate(combined_df, 'collision_rate_run', 'Collision Rate Comparison', 'collision_rate_curve.png', upgrade_episodes)

# --- C. 奖励分布箱线图 (保持原样但增加 hue) ---
plt.figure(figsize=(14, 8))
BOXPLOT_INTERVAL = 500 # 增大间隔让图面更整洁
sampled_df = combined_df[combined_df['episode'] % BOXPLOT_INTERVAL == 0]
sns.boxplot(data=sampled_df, x='episode', y='reward', hue='Condition')
plt.title('Reward Distribution Comparison (Sampled)')
plt.xticks(rotation=45)
plt.savefig(os.path.join(args.output_dir, 'reward_boxplot.png'))
plt.close()

print(f"✅ 所有对比图表已保存至: {args.output_dir}")

In [ ]:
# 压缩并下载训练结果文件夹，包含模型和分析结果。注意路径要根据实际情况调整。
# 1. 压缩真实的实验数据和分析结果
!zip -r /content/experiments_logs_and_models.zip /content/experiments
!zip -r /content/analysis_results_Plz_Converge.zip /content/analysis_results_Plz_Converge

# 2. 触发下载
from google.colab import files

files.download('/content/experiments_logs_and_models.zip')
files.download('/content/analysis_results_Plz_Converge.zip')

In [ ]:
# 三维仿真前置：初始化环境与智能体，并注入权重
import os
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 核心修复：在全局定义 env 和 agent
# ==========================================
# 1. 实例化全局环境对象
env = DroneEnv()

# 2. 实例化全局智能体对象 (使用环境的维度)
state_dim = env.state_dim
action_dim = env.action_dim
max_action = 1.0
agent = TD3(state_dim, action_dim, max_action)

# ==========================================
# 权重加载逻辑 (保持不变)
# ==========================================
target_condition = "Curriculum"
target_seed = "seed_42"
directory = f"./experiments/{target_condition}/{target_seed}"

print(f"🔍 正在扫描指定实验文件夹 {directory} ...")

if not os.path.exists(directory):
    print("❌ 文件夹不存在，请检查路径！")
else:
    pth_files = [f for f in os.listdir(directory) if f.endswith('.pth')]
    actor_file = next((f for f in pth_files if 'actor' in f.lower()), None)

    if actor_file:
        file_path = os.path.join(directory, actor_file)
        try:
            state_dict = torch.load(file_path, map_location=device)
            agent.actor.load_state_dict(state_dict)
            print("✅ 成功加载大脑并初始化环境！")
        except Exception as e:
            print(f"❌ 加载失败: {e}")

In [ ]:
# 三维仿真: 纯净测试
import numpy as np
import torch

# 1. 临时关闭探索噪声，进行“纯净测试”
# 确保你已经实例化了 env 和 agent，并且加载了最好的模型权重
original_noise = agent.policy_noise
agent.policy_noise = 0.0  # 设为0，让它完全按照学到的最优策略飞行

# 2. 初始化环境，准备“黑匣子”记录器
state = env.reset()
done = False

# 记录核心数据
trajectory = []                # 记录飞行的轨迹坐标
start_pos = env.position.copy() # 记录起点
target_pos = env.target.copy()  # 记录终点
obstacles_data = env.obstacles.copy() # 记录障碍物的坐标和半径

# 记录起点
trajectory.append(start_pos.copy())

# 3. 开始闭环飞行测试
print("🚁 无人机起飞，开始纯净飞行测试...")
step_count = 0

while not done:
    # 获取动作 (此时没有随机噪声)
    action = agent.select_action(state)

    # 执行动作
    state, reward, done, info = env.step(action)

    # 记录当前位置
    trajectory.append(env.position.copy())
    step_count += 1

# 转换为 NumPy 数组方便后续画图
trajectory = np.array(trajectory)

# 恢复训练时的噪声设置（好习惯）
agent.policy_noise = original_noise

# 打印最终结果
final_distance = np.linalg.norm(env.target - env.position)
print(f"✅ 飞行结束！共耗时 {step_count} 步。")
if final_distance < 1.5:
    print(f"🎯 成功到达终点！距目标仅 {final_distance:.2f} 米。")
else:
    print(f"💥 发生碰撞或超时。距目标还有 {final_distance:.2f} 米。")

In [ ]:
!pip install plotly
!pip install --upgrade nbformat

In [ ]:
#三维仿真绘图
import plotly.graph_objects as go
import numpy as np

# 1. 初始化 3D 画布
fig = go.Figure()

# 2. 画起点和终点
fig.add_trace(go.Scatter3d(
    x=[start_pos[0]], y=[start_pos[1]], z=[start_pos[2]],
    mode='markers', marker=dict(size=6, color='blue'), name='起点 (Start)'
))
fig.add_trace(go.Scatter3d(
    x=[target_pos[0]], y=[target_pos[1]], z=[target_pos[2]],
    mode='markers', marker=dict(size=8, color='green', symbol='diamond'), name='终点 (Target)'
))

# 3. 画无人机的飞行轨迹线
fig.add_trace(go.Scatter3d(
    x=trajectory[:, 0], y=trajectory[:, 1], z=trajectory[:, 2],
    mode='lines+markers',
    line=dict(color='orange', width=6),
    marker=dict(size=3, color='orange'),
    name='无人机轨迹'
))

# 4. 数学建模：生成 3D 球体表面的网格数据
def create_sphere_mesh(center, radius, resolution=20):
    u = np.linspace(0, 2 * np.pi, resolution)
    v = np.linspace(0, np.pi, resolution)
    x = center[0] + radius * np.outer(np.cos(u), np.sin(v))
    y = center[1] + radius * np.outer(np.sin(u), np.sin(v))
    z = center[2] + radius * np.outer(np.ones(np.size(u)), np.cos(v))
    return x, y, z

# 5. 把所有障碍物画到图中
for i, obs in enumerate(obstacles_data):
    cx, cy, cz = obs['pos']
    r = obs['radius']
    sx, sy, sz = create_sphere_mesh((cx, cy, cz), r)

    fig.add_trace(go.Surface(
        x=sx, y=sy, z=sz,
        colorscale='Reds',      # 红色代表危险
        opacity=0.4,            # 半透明，避免挡住视线
        showscale=False,
        name=f'障碍物 {i+1}'
    ))

# 6. 设置画布的视角和比例
fig.update_layout(
    scene=dict(
        xaxis_title='X (米)',
        yaxis_title='Y (米)',
        zaxis_title='Z (米)',
        # 【关键设置】 aspectmode='data' 强制 XYZ 比例为 1:1:1，确保球体不会变成椭圆
        aspectmode='data'
    ),
    title="无人机 3D 避障航线分析视图",
    margin=dict(l=0, r=0, b=0, t=40),
    legend=dict(x=0.02, y=0.98)
)

# 7. 显示交互图表
fig.show()
# fig.show(renderer="notebook")#适合大多数网页版 Jupyter
# fig.show(renderer="iframe")#适合经典的 Jupyter Notebook
# fig.show(renderer="vscode")#VS Code 的网页版
fig.write_html("drone_3d_flight_Plz_Converge.html")#直接导出为独立 HTML 网页
print("3D仿真图已保存为 drone_3d_flight_Plz_Converge.html！")

In [ ]:
#生成 3D 飞行录像代码
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
# 导入 3D 绘图模块
from mpl_toolkits.mplot3d import Axes3D

# 1. 初始化画布和 3D 坐标轴
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# 2. 画出起点（蓝色）和终点（绿色星号）
ax.scatter(*start_pos, color='blue', s=100, label='Start', edgecolors='black')
ax.scatter(*target_pos, color='green', s=150, marker='*', label='Target', edgecolors='black')

# 3. 绘制所有的障碍物（红色半透明网格球体）
for obs in obstacles_data:
    cx, cy, cz = obs['pos']
    r = obs['radius']
    u = np.linspace(0, 2 * np.pi, 20)
    v = np.linspace(0, np.pi, 20)
    x = cx + r * np.outer(np.cos(u), np.sin(v))
    y = cy + r * np.outer(np.sin(u), np.sin(v))
    z = cz + r * np.outer(np.ones(np.size(u)), np.cos(v))
    # 使用 wireframe 画网格，避免遮挡视线
    ax.plot_wireframe(x, y, z, color='red', alpha=0.2)

# 4. 固定坐标轴的显示范围
# 之前你的代码里撞墙边界是 15，所以我们把视野固定在 [-15, 15]
# 这样可以防止动画播放时镜头跟着无人机乱晃
ax.set_xlim([-15, 15])
ax.set_ylim([-15, 15])
ax.set_zlim([-15, 15])
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('Drone 3D Flight Animation')
ax.legend()

# 5. 初始化动画中的“动态元素”：飞行轨迹线 和 无人机当前位置点
trail_line, = ax.plot([], [], [], color='orange', linewidth=2, label='Trajectory')
drone_point, = ax.plot([], [], [], 'o', color='red', markersize=8, label='Drone')

# 动画初始化函数
def init():
    trail_line.set_data([], [])
    trail_line.set_3d_properties([])
    drone_point.set_data([], [])
    drone_point.set_3d_properties([])
    return trail_line, drone_point

# 动画的每一帧更新函数
def update(frame):
    # 获取从第 0 帧到当前帧的所有历史轨迹
    current_traj = trajectory[:frame+1]

    # 更新轨迹线
    trail_line.set_data(current_traj[:, 0], current_traj[:, 1])
    trail_line.set_3d_properties(current_traj[:, 2])

    # 更新无人机当前的红点位置
    current_pos = trajectory[frame]
    drone_point.set_data([current_pos[0]], [current_pos[1]])
    drone_point.set_3d_properties([current_pos[2]])

    return trail_line, drone_point

print("⏳ 正在全力渲染 3D 逐帧动画，请稍候（可能需要几十秒）...")

# 6. 生成动画！(interval=50 表示每帧间隔 50 毫秒，即 20 FPS)
# frames 取轨迹的总步数
ani = animation.FuncAnimation(fig, update, frames=len(trajectory),
                              init_func=init, blit=False, interval=50)

# 7. 关闭静态多余的图表显示，并将动画转为 HTML5 交互式播放器
plt.close()
HTML(ani.to_jshtml())